In [ ]:
from google.colab import files
updated = files.upload()

import zipfile
import io


for fn in updated.keys():
  if fn.endswith('.zip'):
    zip_ref = zipfile.ZipFile(io.BytesIO(updated[fn]), 'r')
    zip_ref.extractall('/') # Extract to the root directory for easy access
    zip_ref.close()

import pandas as pd
# Now, read the extracted CSV file
df = pd.read_csv('/mtsamples.csv')
df

In [ ]:
# Load and explore the data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Load data
df = pd.read_csv('/mtsamples.csv', index_col=0)
print(df.shape)
print(df.head())
print(df.columns.tolist())
print(df.dtypes)

In [ ]:
# Understand the key columns

# The columns you care about most:
# 'medical_specialty' — the label you're predicting
# 'transcription' — the clinical note text (your input)
# 'description' — brief summary of the note

print(df['medical_specialty'].value_counts())
print(f"\nNumber of specialties: {df['medical_specialty'].nunique()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

In [ ]:
# Visualise the specialty distribution

import os

# Count notes per specialty
specialty_counts = df['medical_specialty'].value_counts()

plt.figure(figsize=(14, 8))
specialty_counts.plot(kind='barh')
plt.xlabel('Number of Notes')
plt.title('Distribution of Clinical Notes by Medical Specialty')
plt.tight_layout()

# Create the 'images' directory if it doesn't exist
output_dir = 'images'
os.makedirs(output_dir, exist_ok=True)

plt.savefig(os.path.join(output_dir, 'specialty_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTop 5 specialties:\n{specialty_counts.head()}")
print(f"\nBottom 5 specialties:\n{specialty_counts.tail()}")

In [ ]:
# Handle the class imbalance problem

# Keep only top 10 specialties
top_10 = specialty_counts.head(10).index.tolist()
df_filtered = df[df['medical_specialty'].isin(top_10)].copy()

print(f"Original dataset: {len(df)} notes")
print(f"Filtered dataset: {len(df_filtered)} notes")
print(f"\nSpecialties kept:\n{df_filtered['medical_specialty'].value_counts()}")

# Drop rows with missing transcription text
df_filtered = df_filtered.dropna(subset=['transcription'])
print(f"\nAfter dropping missing transcriptions: {len(df_filtered)} notes")

## Text Preprocessing

In [ ]:
# Install NLP libraries

!pip install nltk scikit-learn wordcloud
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [ ]:
# Write a text cleaning function

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Add medical stopwords that appear everywhere and add no signal
medical_stopwords = {'patient', 'procedure', 'history', 'diagnosis',
                     'noted', 'placed', 'used', 'performed', 'right',
                     'left', 'including', 'mg', 'also', 'well'}
stop_words.update(medical_stopwords)

def clean_text(text):
    """Clean and preprocess clinical note text"""
    if pd.isna(text):
        return ""

    # Lowercase
    text = text.lower()

    # Remove numbers and special characters
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenise (split into words)
    tokens = text.split()

    # Remove stopwords and lemmatise (reduce words to root form)
    # e.g. "bleeding", "bled", "bleeds" → "bleed"
    tokens = [lemmatizer.lemmatize(word) for word in tokens
              if word not in stop_words and len(word) > 2]

    return ' '.join(tokens)

# Apply to dataset
df_filtered['cleaned_text'] = df_filtered['transcription'].apply(clean_text)
print(df_filtered['cleaned_text'].head(3))

In [ ]:
# Visualise most common words per specialty (Word Cloud)

from wordcloud import WordCloud

# Find a specialty with non-empty cleaned text to generate a word cloud
specialty_to_visualize = None
for s in df_filtered['medical_specialty'].unique():
    temp_text = ' '.join(df_filtered[
        df_filtered['medical_specialty'] == s]['cleaned_text'])
    if temp_text: # If temp_text is not empty, we found a suitable specialty
        specialty_to_visualize = s
        break

if specialty_to_visualize:
    print(f"Generating word cloud for specialty: {specialty_to_visualize}")
    specialty_text = ' '.join(df_filtered[
        df_filtered['medical_specialty'] == specialty_to_visualize]['cleaned_text'])

    wordcloud = WordCloud(width=800, height=400,
                          background_color='white').generate(specialty_text)

    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Most Common Words in {specialty_to_visualize} Notes')
    plt.savefig('images/wordcloud.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Could not find any specialty with non-empty cleaned text to generate a word cloud.")
    print("Consider reviewing the 'clean_text' function, the stopwords, or the filtered dataset.")

In [ ]:
# Baseline Model

# encode labels and split data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode specialty labels as numbers
le = LabelEncoder()
df_filtered['label'] = le.fit_transform(df_filtered['medical_specialty'])

print("Label mapping:")
for i, specialty in enumerate(le.classes_):
    print(f"  {i}: {specialty}")

X = df_filtered['cleaned_text']
y = df_filtered['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
#  Build TF-IDF + Logistic Regression pipeline

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# A Pipeline chains steps together — first vectorise, then classify
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42,
                                class_weight='balanced'))
])

tfidf_pipeline.fit(X_train, y_train)
y_pred_tfidf = tfidf_pipeline.predict(X_test)

print(f"TF-IDF + Logistic Regression Accuracy: "
      f"{accuracy_score(y_test, y_pred_tfidf):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tfidf,
                            target_names=le.classes_))

In [ ]:
# Also try Naive Bayes and SVM

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

models = {
    'Logistic Regression': tfidf_pipeline,
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000)),
        ('clf', MultinomialNB())
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=10000)),
        ('clf', LinearSVC(random_state=42, class_weight='balanced'))
    ])
}

results = {}
for name, model in models.items():
    if name != 'Logistic Regression':  # Already trained above
        model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = {'model': model, 'predictions': preds, 'accuracy': acc}
    print(f"{name}: {acc:.4f}")

In [ ]:
# C0nfusion matrix for best model
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_preds = results[best_name]['predictions']

cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('images/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Install HuggingFace Transformers
!pip install transformers torch datasets

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch
from torch.utils.data import Dataset

In [ ]:
# Load BioBERT tokeniser
# BioBERT is pre-trained specifically on biomedical text
model_name = "dmis-lab/biobert-base-cased-v1.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Test the tokeniser
sample = df_filtered['cleaned_text'].iloc[0]
tokens = tokenizer(sample, truncation=True, max_length=512)
print(f"Sample text: {sample[:100]}...")
print(f"Number of tokens: {len(tokens['input_ids'])}")

In [ ]:
# Create a PyTorch Dataset
class ClinicalNotesDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
train_dataset = ClinicalNotesDataset(X_train.reset_index(drop=True),
                                      y_train.reset_index(drop=True),
                                      tokenizer)
test_dataset = ClinicalNotesDataset(X_test.reset_index(drop=True),
                                     y_test.reset_index(drop=True),
                                     tokenizer)

In [ ]:
# Load and fine-tune BioBERT

from sklearn.metrics import accuracy_score as sk_accuracy
import numpy as np

num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels)

# Training configuration
training_args = TrainingArguments(
    output_dir='./biobert_results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch", # Corrected from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': sk_accuracy(labels, predictions)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Train — this will take 15–30 minutes on Colab GPU
# Make sure to enable GPU: Runtime → Change runtime type → T4 GPU
trainer.train()

In [ ]:
# Evaluate BioBERT

# Get predictions
predictions = trainer.predict(test_dataset)
y_pred_biobert = np.argmax(predictions.predictions, axis=-1)

biobert_acc = sk_accuracy(y_test, y_pred_biobert)
print(f"BioBERT Accuracy: {biobert_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_biobert,
                            target_names=le.classes_))

In [ ]:
# Classification report Heatmap
from sklearn.metrics import classification_report
import pandas as pd

# Build a heatmap from the classification report
report = classification_report(y_test, y_pred_biobert,
                                target_names=le.classes_,
                                output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df = report_df.iloc[:-3, :-1]  # Remove summary rows

plt.figure(figsize=(10, 8))
sns.heatmap(report_df.astype(float), annot=True, fmt='.2f',
            cmap='YlOrRd', vmin=0, vmax=1)
plt.title('BioBERT Classification Report Heatmap')
plt.tight_layout()
plt.savefig('images/classification_report_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final model comparison summary

print("=" * 50)
print("FINAL MODEL COMPARISON")
print("=" * 50)
for name, res in results.items():
    print(f"{name}: {res['accuracy']:.4f}")
print(f"BioBERT: {biobert_acc:.4f}")
print("=" * 50)
print(f"Best model: BioBERT")
print(f"Improvement over baseline: "
      f"{(biobert_acc - max(r['accuracy'] for r in results.values())):.4f}")